# Prediksi Curah Hujan Multi-Skala Waktu — XGBoost v2
**Fokus**: Two-Stage Prediction (Klasifikasi + Regresi), Multi-Skala Waktu (1 Jam, 3 Jam, Harian), Bayesian Optimization (Optuna), SHAP Feature Importance.

## Tujuan
Notebook ini mengimplementasikan sistem prediksi curah hujan hierarkis dua tahap menggunakan XGBoost untuk tiga resolusi temporal secara berurutan:
- **Model A**: Prediksi 1 jam ke depan
- **Model B**: Prediksi 3 jam ke depan
- **Model C**: Prediksi harian (24 jam) ke depan

Setiap model menggunakan pendekatan dua tahap:
- **Tahap 1**: Klasifikasi kejadian hujan (P(hujan))
- **Tahap 2**: Estimasi jumlah curah hujan (mm) — hanya jika hujan diprediksi terjadi

In [1]:
!pip install --upgrade shap optuna xgboost scikit-learn --quiet

import os, sys, random, json, logging, warnings, datetime
warnings.filterwarnings('ignore')
from pathlib import Path
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import xgboost as xgb
import shap

from sklearn.preprocessing import MinMaxScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, confusion_matrix, log_loss,
    mean_squared_error, mean_absolute_error, r2_score,
    precision_recall_curve, roc_curve, auc, average_precision_score
)
from sklearn.calibration import calibration_curve

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

# ============================================================
# KONFIGURASI UTAMA — Ubah TEST_MODE=False untuk Kaggle
# ============================================================
TEST_MODE = False

CONFIG = {
    'TEST_MODE': TEST_MODE,
    'TEST_ROWS': 2000,
    'N_TRIALS': 5 if TEST_MODE else 30,
    'SEED': 42,
    'TRAIN_YEARS': (2005, 2023),
    'VAL_YEAR': 2024,
    'TEST_YEAR': 2025,
}

def is_kaggle():
    return os.path.exists('/kaggle/input')

if is_kaggle():
    OUTPUT_BASE = Path('/kaggle/working/outputs/xgboost')
else:
    OUTPUT_BASE = Path(r'D:\Github\Projek_Rainfall\DeepLearning_Meteorologi\outputs\xgboost')

for scale in ['1h', '3h', 'daily']:
    for subdir in ['plots', 'metrics', 'shap', 'predictions']:
        (OUTPUT_BASE / scale / subdir).mkdir(parents=True, exist_ok=True)

seed_everything(CONFIG['SEED'])
logger.info(f"Mode: {'TEST' if TEST_MODE else 'FULL'} | Output: {OUTPUT_BASE}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 498.0/498.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


2026-07-11 16:00:44,201 - INFO - Mode: FULL | Output: /kaggle/working/outputs/xgboost


## Pemuatan Data
Memuat data cuaca dari sumber CSV dan memilih kolom yang relevan untuk analisis.

In [2]:
def get_paths():
    if is_kaggle():
        return Path('/kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/open_meteo_jerukagung/cuaca_jerukagung.csv')
    return Path(r'D:\Github\Projek_Rainfall\Analisis_Meteorologi\open_meteo_jerukagung\cuaca_jerukagung.csv')

ESSENTIAL_COLS = [
    'rain', 'temperature_2m', 'wet_bulb_temperature_2m', 'relative_humidity_2m',
    'dew_point_2m', 'total_column_integrated_water_vapour', 'surface_pressure',
    'pressure_msl', 'wind_speed_10m', 'wind_gusts_10m', 'wind_direction_10m',
    'cloud_cover', 'cloud_cover_high', 'cloud_cover_mid', 'cloud_cover_low',
    'boundary_layer_height', 'vapour_pressure_deficit', 'sunshine_duration',
    'shortwave_radiation', 'direct_radiation', 'diffuse_radiation',
    'direct_normal_irradiance', 'et0_fao_evapotranspiration'
]

def load_data(filepath):
    logger.info(f"Memuat data dari {filepath}")
    df = pd.read_csv(filepath)
    for col in ['datetime', 'date']:
        if col in df.columns:
            df = df.set_index(col)
            break
    df.index = pd.to_datetime(df.index, utc=True).tz_convert('Asia/Jakarta').tz_localize(None)
    df.index.name = 'date'
    df = df.sort_index()
    cols = [c for c in ESSENTIAL_COLS if c in df.columns]
    df = df[cols]
    if 'rain' in df.columns:
        df.loc[df['rain'] < 0, 'rain'] = 0
    if CONFIG['TEST_MODE']:
        df = df.tail(CONFIG['TEST_ROWS'])
        logger.info(f"TEST_MODE: menggunakan {len(df)} baris terakhir")
    logger.info(f"Data dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom | {df.index.min()} s/d {df.index.max()}")
    return df

data_path = get_paths()
df_raw = load_data(data_path)
display(df_raw.head())


2026-07-11 16:00:44,230 - INFO - Memuat data dari /kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/open_meteo_jerukagung/cuaca_jerukagung.csv
2026-07-11 16:00:46,028 - INFO - Data dimuat: 232,488 baris, 23 kolom | 2000-01-01 00:00:00 s/d 2026-07-09 23:00:00


,rain,temperature_2m,wet_bulb_temperature_2m,relative_humidity_2m,dew_point_2m,total_column_integrated_water_vapour,surface_pressure,pressure_msl,wind_speed_10m,wind_gusts_10m,...,cloud_cover_mid,cloud_cover_low,boundary_layer_height,vapour_pressure_deficit,sunshine_duration,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance,et0_fao_evapotranspiration
date,,,,,,,,,,,,,,,,,,,,,
2000-01-01 00:00:00,0.0,25.304500,24.187763,91.417620,23.804500,57.1,1005.71070,1007.9,3.415260,5.40,...,10.0,6.0,45.0,0.276608,0.0,0.0,0.0,0.0,0.0,0.0
2000-01-01 01:00:00,0.0,24.504500,24.269080,97.924580,24.154501,57.7,1005.50540,1007.7,2.811690,5.40,...,6.0,6.0,35.0,0.063775,0.0,0.0,0.0,0.0,0.0,0.0
2000-01-01 02:00:00,0.0,24.304500,24.186255,98.807600,24.104500,57.6,1005.00494,1007.2,2.902413,6.84,...,5.0,27.0,40.0,0.036206,0.0,0.0,0.0,0.0,0.0,0.0
2000-01-01 03:00:00,0.5,23.904501,23.823399,99.101860,23.754500,56.9,1004.60300,1006.8,2.036468,4.68,...,7.0,22.0,45.0,0.026627,0.0,0.0,0.0,0.0,0.0,0.0
2000-01-01 04:00:00,0.0,24.004500,23.884535,98.804955,23.804500,56.2,1004.40405,1006.6,2.811690,6.12,...,6.0,22.0,55.0,0.035641,0.0,0.0,0.0,0.0,0.0,0.0


## Rekayasa Fitur
Menghasilkan fitur turunan dari data mentah: dekomposisi angin, tren atmosfer, fitur siklik, dan indikator fisika cuaca.

In [3]:
def generate_features(df):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # 1. Dekomposisi vektor angin (U dan V)
    if 'wind_speed_10m' in df.columns and 'wind_direction_10m' in df.columns:
        wd_rad = df['wind_direction_10m'] * np.pi / 180.0
        df['wind_u'] = -df['wind_speed_10m'] * np.sin(wd_rad)
        df['wind_v'] = -df['wind_speed_10m'] * np.cos(wd_rad)
        df = df.drop(columns=['wind_speed_10m', 'wind_direction_10m'])

    # 2. Tren atmosfer: laju perubahan 3 jam terakhir
    trend_cols = [c for c in ['temperature_2m', 'relative_humidity_2m',
                               'pressure_msl', 'surface_pressure'] if c in df.columns]
    for col in trend_cols:
        df[f'{col}_change_3h'] = df[col].diff(1)
    if 'wind_u' in df.columns:
        df['wind_u_change_3h'] = df['wind_u'].diff(1)
        df['wind_v_change_3h'] = df['wind_v'].diff(1)

    # 3. Dew Point Depression (indikator kejenuhan udara)
    if 'temperature_2m' in df.columns and 'relative_humidity_2m' in df.columns:
        df['dew_point_depression'] = (100 - df['relative_humidity_2m']) / 5

    # 4. Fitur siklik (jam dan bulan)
    df['hour_sin']  = np.sin(2 * np.pi * df.index.hour / 24.0)
    df['hour_cos']  = np.cos(2 * np.pi * df.index.hour / 24.0)
    df['month_sin'] = np.sin(2 * np.pi * df.index.month / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * df.index.month / 12.0)

    df = df.dropna()
    return df

df_feat = generate_features(df_raw)
logger.info(f"Fitur dihasilkan: {df_feat.shape[1]} kolom")
df_feat.info()


2026-07-11 16:00:46,178 - INFO - Fitur dihasilkan: 34 kolom


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 228006 entries, 2000-01-01 01:00:00 to 2026-07-05 06:00:00
Data columns (total 34 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   rain                                  228006 non-null  float64
 1   temperature_2m                        228006 non-null  float64
 2   wet_bulb_temperature_2m               228006 non-null  float64
 3   relative_humidity_2m                  228006 non-null  float64
 4   dew_point_2m                          228006 non-null  float64
 5   total_column_integrated_water_vapour  228006 non-null  float64
 6   surface_pressure                      228006 non-null  float64
 7   pressure_msl                          228006 non-null  float64
 8   wind_gusts_10m                        228006 non-null  float64
 9   cloud_cover                           228006 non-null  float64
 10  cloud_cover_high                  

## Persiapan Target Multi-Skala
Mendefinisikan target prediksi dan threshold kejadian hujan untuk setiap skala waktu.

In [4]:
SCALE_CONFIG = {
    '1h': {
        'resample': None,        # Tidak di-resample (sudah per jam)
        'shift': -1,             # Target = jam berikutnya
        'threshold': 0.5,        # Hujan jika >= 0.5 mm/jam
        'label': 'Prediksi 1 Jam',
        'unit': 'mm/jam',
    },
    '3h': {
        'resample': '3h',
        'shift': -1,             # Target = 3 jam berikutnya
        'threshold': 1.0,        # Hujan jika >= 1.0 mm/3jam
        'label': 'Prediksi 3 Jam',
        'unit': 'mm/3jam',
    },
    'daily': {
        'resample': 'D',
        'shift': -1,             # Target = 24 jam berikutnya
        'threshold': 1.0,        # Hujan jika >= 1.0 mm/hari
        'label': 'Prediksi Harian',
        'unit': 'mm/hari',
    },
}

def prepare_targets(df_feat, scale):
    scfg = SCALE_CONFIG[scale]
    df = df_feat.copy()

    if scfg['resample'] is not None:
        agg_rules = {c: 'mean' for c in df.columns}
        if 'rain' in df.columns:
            agg_rules['rain'] = 'sum'
        df = df.resample(scfg['resample']).agg(agg_rules).dropna()

    df['target_amount'] = df['rain'].shift(scfg['shift'])
    df = df.dropna()
    thr = scfg['threshold']
    df['target_occurrence'] = (df['target_amount'] >= thr).astype(int)

    occ_rate = df['target_occurrence'].mean()
    logger.info(f"[{scale}] Dataset: {len(df):,} baris | "
                f"Kejadian hujan: {occ_rate:.1%} | "
                f"Threshold: >={thr} {scfg['unit']}")
    return df


## Fungsi Metrik Evaluasi
Fungsi untuk menghitung metrik klasifikasi (CSI, POD, FAR, ETS, HSS, ROC-AUC, dll) dan regresi (RMSE, MAE, NSE, KGE, dll).

In [5]:
def met_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    hits             = np.sum((y_pred == 1) & (y_true == 1))
    misses           = np.sum((y_pred == 0) & (y_true == 1))
    false_alarms     = np.sum((y_pred == 1) & (y_true == 0))
    correct_negatives = np.sum((y_pred == 0) & (y_true == 0))
    total = hits + misses + false_alarms + correct_negatives

    csi = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else 0
    pod = hits / (hits + misses) if (hits + misses) > 0 else 0
    far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else 0
    hits_r = ((hits + misses) * (hits + false_alarms)) / total if total > 0 else 0
    ets = (hits - hits_r) / (hits + misses + false_alarms - hits_r) if (hits + misses + false_alarms - hits_r) > 0 else 0
    denom = ((hits+misses)*(misses+correct_negatives) + (hits+false_alarms)*(false_alarms+correct_negatives))
    hss = (2*(hits*correct_negatives - misses*false_alarms)) / denom if denom > 0 else 0
    return {'CSI': csi, 'POD': pod, 'FAR': far, 'ETS': ets, 'HSS': hss}

def compute_classification_metrics(y_true, prob_uncal, prob_cal, pred_cal):
    y_true = np.asarray(y_true)
    met = met_metrics(y_true, pred_cal)
    pr, rc, _ = precision_recall_curve(y_true, prob_cal)
    pr_auc = auc(rc, pr)
    return {
        'Accuracy':    accuracy_score(y_true, pred_cal),
        'Precision':   precision_score(y_true, pred_cal, zero_division=0),
        'Recall':      recall_score(y_true, pred_cal, zero_division=0),
        'F1':          f1_score(y_true, pred_cal, zero_division=0),
        'ROC_AUC':     roc_auc_score(y_true, prob_cal) if len(np.unique(y_true)) > 1 else 0.0,
        'PR_AUC':      pr_auc,
        'Brier_Uncal': brier_score_loss(y_true, prob_uncal),
        'Brier_Cal':   brier_score_loss(y_true, prob_cal),
        **met
    }

def compute_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) == 0:
        return {k: np.nan for k in ['MAE','RMSE','R2','NSE','KGE','Bias','Correlation']}
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    nse  = 1 - ss_res / (ss_tot + 1e-10)
    r_p  = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else 0
    alpha = np.std(y_pred) / (np.std(y_true) + 1e-10)
    beta  = np.mean(y_pred) / (np.mean(y_true) + 1e-10)
    kge  = 1 - np.sqrt((r_p-1)**2 + (alpha-1)**2 + (beta-1)**2)
    bias = np.mean(y_pred - y_true)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'NSE': nse, 'KGE': kge,
            'Bias': bias, 'Correlation': r_p}


## Fungsi Visualisasi
Fungsi untuk menghasilkan setiap jenis plot secara terpisah dan menyimpannya sebagai file PNG.

In [6]:
import matplotlib.ticker as mticker

def _add_timestamp(ax):
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    ax.annotate(f'Dibuat: {ts}', xy=(1, 0), xycoords='axes fraction',
                fontsize=7, ha='right', va='bottom',
                color='gray', style='italic')

def save_fig(fig, path):
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    logger.info(f"Plot disimpan: {path}")

def plot_reliability_diagram(y_true, prob_uncal, prob_iso, prob_platt, out_dir, scale):
    fig, axes = plt.subplots(1, 1, figsize=(7, 6))
    ax = axes
    f_iso,   m_iso   = calibration_curve(y_true, prob_iso,   n_bins=10)
    f_platt, m_platt = calibration_curve(y_true, prob_platt, n_bins=10)
    f_uncal, m_uncal = calibration_curve(y_true, prob_uncal, n_bins=10)
    ax.plot([0,1],[0,1], 'k:', label='Kalibrasi Sempurna')
    ax.plot(m_uncal, f_uncal, 's--', alpha=0.6, label='Tidak Terkalibrasi')
    ax.plot(m_iso,   f_iso,   'o-',  label='Isotonic')
    ax.plot(m_platt, f_platt, '^-',  label='Platt Scaling')
    ax.set_xlabel('Probabilitas Prediksi Rata-rata', fontsize=12)
    ax.set_ylabel('Fraksi Kejadian Positif', fontsize=12)
    ax.set_title(f'Diagram Reliabilitas — {SCALE_CONFIG[scale]["label"]}\nXGBoost Two-Stage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'reliability_diagram.png')

def plot_probability_distribution(prob_uncal, prob_iso, prob_platt, out_dir, scale):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(prob_uncal, color='red',   alpha=0.35, label='Tidak Terkalibrasi', kde=True, bins=30, ax=ax)
    sns.histplot(prob_iso,   color='blue',  alpha=0.35, label='Isotonic',           kde=True, bins=30, ax=ax)
    sns.histplot(prob_platt, color='green', alpha=0.35, label='Platt Scaling',      kde=True, bins=30, ax=ax)
    ax.set_xlabel('Probabilitas Hujan', fontsize=12)
    ax.set_ylabel('Frekuensi', fontsize=12)
    ax.set_title(f'Distribusi Probabilitas — {SCALE_CONFIG[scale]["label"]}\nXGBoost Two-Stage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'probability_distribution.png')

def plot_confusion_matrix(y_true, y_pred, out_dir, scale):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Tidak Hujan', 'Hujan'],
                yticklabels=['Tidak Hujan', 'Hujan'])
    ax.set_xlabel('Prediksi', fontsize=12)
    ax.set_ylabel('Aktual', fontsize=12)
    ax.set_title(f'Matriks Kebingungan — {SCALE_CONFIG[scale]["label"]}\nXGBoost Two-Stage', fontsize=13, fontweight='bold')
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'confusion_matrix.png')

def plot_roc_curve(y_true, prob_cal, out_dir, scale):
    fpr, tpr, _ = roc_curve(y_true, prob_cal)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
    ax.plot([0,1],[0,1],'k--', alpha=0.5)
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'Kurva ROC — {SCALE_CONFIG[scale]["label"]}\nXGBoost Two-Stage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'roc_curve.png')

def plot_precision_recall_curve(y_true, prob_cal, out_dir, scale):
    pr, rc, _ = precision_recall_curve(y_true, prob_cal)
    pr_auc = auc(rc, pr)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(rc, pr, lw=2, label=f'PR (AUC = {pr_auc:.3f})')
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title(f'Kurva Precision-Recall — {SCALE_CONFIG[scale]["label"]}\nXGBoost Two-Stage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'precision_recall_curve.png')

def plot_prediction_vs_observation(y_true, y_pred, out_dir, scale):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    max_val = max(y_true.max(), y_pred.max()) if len(y_true) > 0 else 1
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(y_true, y_pred, alpha=0.3, s=15)
    ax.plot([0, max_val],[0, max_val], 'r--', label='Ideal (y=x)')
    ax.set_xlabel(f'Observasi ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_ylabel(f'Prediksi ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_title(f'Prediksi vs Observasi — {SCALE_CONFIG[scale]["label"]}\nXGBoost Two-Stage (Hanya Saat Hujan)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'prediction_vs_observation.png')

def plot_residual_distribution(y_true, y_pred, out_dir, scale):
    residuals = np.asarray(y_pred) - np.asarray(y_true)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(residuals, kde=True, bins=30, ax=ax)
    ax.axvline(0, color='r', linestyle='--', label='Bias=0')
    ax.set_xlabel(f'Residu (Prediksi - Observasi) [{SCALE_CONFIG[scale]["unit"]}]', fontsize=12)
    ax.set_ylabel('Frekuensi', fontsize=12)
    ax.set_title(f'Distribusi Residu — {SCALE_CONFIG[scale]["label"]}\nXGBoost Two-Stage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'residual_distribution.png')

def plot_timeseries_prediction(y_true, y_pred, index, out_dir, scale, n_show=200):
    if len(y_true) > n_show:
        y_true = y_true[-n_show:]
        y_pred = y_pred[-n_show:]
        index  = index[-n_show:]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(index, y_true, label='Observasi', linewidth=1.5, color='steelblue')
    ax.plot(index, y_pred, label='Prediksi',  linewidth=1.5, color='orangered', alpha=0.85)
    ax.set_xlabel('Waktu', fontsize=12)
    ax.set_ylabel(f'Curah Hujan ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_title(f'Deret Waktu Prediksi vs Observasi — {SCALE_CONFIG[scale]["label"]}\nXGBoost Two-Stage ({n_show} titik terakhir)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'timeseries_prediction_vs_observation.png')

def plot_shap_importance(shap_values, feature_names, out_dir, scale, title_suffix=''):
    shap_mean = np.abs(shap_values).mean(axis=0)
    idx = np.argsort(shap_mean)[::-1][:20]
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(range(len(idx)), shap_mean[idx][::-1])
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels([feature_names[i] for i in idx[::-1]], fontsize=9)
    ax.set_xlabel('Nilai SHAP Rata-rata |SHAP|', fontsize=12)
    ax.set_title(f'Kepentingan Fitur SHAP — {SCALE_CONFIG[scale]["label"]}\nXGBoost {title_suffix}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    _add_timestamp(ax)
    save_fig(fig, out_dir / f'shap_importance_{title_suffix.replace(" ","_").lower()}.png')

def plot_gain_importance(clf, feature_names, out_dir, scale, title_suffix=''):
    gain = clf.get_booster().get_score(importance_type='gain')
    feat_names = list(gain.keys())
    scores = list(gain.values())
    idx = np.argsort(scores)[::-1][:20]
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(range(len(idx)), [scores[i] for i in idx][::-1])
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels([feat_names[i] for i in idx[::-1]], fontsize=9)
    ax.set_xlabel('Gain Importance', fontsize=12)
    ax.set_title(f'Kepentingan Fitur (Gain) — {SCALE_CONFIG[scale]["label"]}\nXGBoost {title_suffix}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    _add_timestamp(ax)
    save_fig(fig, out_dir / f'gain_importance_{title_suffix.replace(" ","_").lower()}.png')

# --- METEOROLOGICAL VALIDATION PLOTS ADDED BY AGENT ---
def categorize_rainfall(amount, scale):
    amount = np.asarray(amount)
    cats = np.zeros(amount.shape, dtype=object)
    if scale == '1h':
        cats[amount < 0.1] = 'No Rain (<0.1)'
        cats[(amount >= 0.1) & (amount < 2.0)] = 'Light (0.1-2.0)'
        cats[(amount >= 2.0) & (amount < 5.0)] = 'Moderate (2.0-5.0)'
        cats[(amount >= 5.0) & (amount < 10.0)] = 'Heavy (5.0-10.0)'
        cats[amount >= 10.0] = 'Very Heavy (>=10)'
    elif scale == '3h':
        cats[amount < 0.5] = 'No Rain (<0.5)'
        cats[(amount >= 0.5) & (amount < 5.0)] = 'Light (0.5-5.0)'
        cats[(amount >= 5.0) & (amount < 15.0)] = 'Moderate (5.0-15.0)'
        cats[(amount >= 15.0) & (amount < 30.0)] = 'Heavy (15.0-30.0)'
        cats[amount >= 30.0] = 'Very Heavy (>=30)'
    else:  # 'daily'
        cats[amount < 0.5] = 'No Rain (<0.5)'
        cats[(amount >= 0.5) & (amount < 20.0)] = 'Light (0.5-20)'
        cats[(amount >= 20.0) & (amount < 50.0)] = 'Moderate (20-50)'
        cats[(amount >= 50.0) & (amount < 100.0)] = 'Heavy (50-100)'
        cats[amount >= 100.0] = 'Very Heavy (>=100)'
    return cats

def plot_meteorological_confusion_matrix(y_true, y_pred, out_dir, scale):
    y_true_cat = categorize_rainfall(y_true, scale)
    y_pred_cat = categorize_rainfall(y_pred, scale)
    
    no_rain_lbl = 'No Rain (<0.1)' if scale == '1h' else 'No Rain (<0.5)'
    light_lbl = 'Light (0.1-2.0)' if scale == '1h' else 'Light (0.5-5.0)' if scale == '3h' else 'Light (0.5-20)'
    mod_lbl = 'Moderate (2.0-5.0)' if scale == '1h' else 'Moderate (5.0-15.0)' if scale == '3h' else 'Moderate (20-50)'
    heavy_lbl = 'Heavy (5.0-10.0)' if scale == '1h' else 'Heavy (15.0-30.0)' if scale == '3h' else 'Heavy (50-100)'
    vheavy_lbl = 'Very Heavy (>=10)' if scale == '1h' else 'Very Heavy (>=30)' if scale == '3h' else 'Very Heavy (>=100)'
    
    order = [no_rain_lbl, light_lbl, mod_lbl, heavy_lbl, vheavy_lbl]
    
    from sklearn.metrics import confusion_matrix as sk_confusion_matrix
    cm = sk_confusion_matrix(y_true_cat, y_pred_cat, labels=order)
    
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax,
                xticklabels=[c.split(' (')[0] for c in order],
                yticklabels=[c.split(' (')[0] for c in order])
    ax.set_xlabel('Prediksi Kategori BMKG', fontsize=12)
    ax.set_ylabel('Aktual Kategori BMKG', fontsize=12)
    ax.set_title(f'Matriks Kebingungan Meteorologi BMKG — {SCALE_CONFIG[scale]["label"]}', fontsize=13, fontweight='bold')
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'meteorological_confusion_matrix.png')

def plot_csi_vs_threshold(y_true, y_pred, out_dir, scale):
    if scale == '1h':
        thresholds = [0.1, 0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0]
    elif scale == '3h':
        thresholds = [0.5, 1.0, 2.0, 3.0, 5.0, 10.0, 15.0, 20.0]
    else:  # 'daily'
        thresholds = [0.5, 1.0, 5.0, 10.0, 20.0, 30.0, 50.0, 75.0]
        
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    csi_scores = []
    
    for t in thresholds:
        hit = np.sum((y_true >= t) & (y_pred >= t))
        fa = np.sum((y_true < t) & (y_pred >= t))
        miss = np.sum((y_true >= t) & (y_pred < t))
        denominator = hit + fa + miss
        csi = hit / denominator if denominator > 0 else 0.0
        csi_scores.append(csi)
        
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(thresholds, csi_scores, 'o-', linewidth=2, color='darkviolet', label='CSI (Threat Score)')
    ax.set_xlabel(f'Threshold Curah Hujan ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_ylabel('CSI Score', fontsize=12)
    ax.set_title(f'CSI (Threat Score) vs. Threshold Hujan — {SCALE_CONFIG[scale]["label"]}', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'csi_vs_threshold.png')

def plot_hexbin_prediction_vs_observation(y_true, y_pred, out_dir, scale):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if len(y_true) == 0:
        return
    max_val = max(y_true.max(), y_pred.max()) if len(y_true) > 0 else 1
    
    fig, ax = plt.subplots(figsize=(8, 7))
    hb = ax.hexbin(y_true, y_pred, gridsize=30, cmap='YlOrRd', mincnt=1, bins='log')
    cb = fig.colorbar(hb, ax=ax, label='Log10(Frekuensi)')
    ax.plot([0, max_val], [0, max_val], 'b--', alpha=0.7, label='Ideal (y=x)')
    ax.set_xlabel(f'Observasi ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_ylabel(f'Prediksi ({SCALE_CONFIG[scale]["unit"]})', fontsize=12)
    ax.set_title(f'Kepadatan Prediksi vs Observasi (Hexbin) — {SCALE_CONFIG[scale]["label"]}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    _add_timestamp(ax)
    save_fig(fig, out_dir / 'prediction_vs_observation_hexbin.png')


## Fungsi Ekspor Hasil
Fungsi untuk menyimpan metrik, prediksi, dan hyperparameter ke dalam file CSV dan JSON.

In [7]:
def save_classification_metrics(metrics_dict, out_dir, scale, model_name='xgboost'):
    df = pd.DataFrame([metrics_dict])
    df.insert(0, 'model', model_name)
    df.insert(1, 'scale', scale)
    path = out_dir / 'classification_metrics.csv'
    df.to_csv(path, index=False)
    logger.info(f"Metrik klasifikasi disimpan: {path}")
    return df

def save_regression_metrics(metrics_dict, out_dir, scale, model_name='xgboost'):
    df = pd.DataFrame([metrics_dict])
    df.insert(0, 'model', model_name)
    df.insert(1, 'scale', scale)
    path = out_dir / 'regression_metrics.csv'
    df.to_csv(path, index=False)
    logger.info(f"Metrik regresi disimpan: {path}")
    return df

def save_predictions(y_true_occ, prob_cal, pred_occ, y_true_reg, y_pred_reg, index, out_dir):
    df = pd.DataFrame({
        'timestamp': index,
        'y_true_occurrence': y_true_occ,
        'prob_rain_calibrated': prob_cal,
        'pred_occurrence': pred_occ,
    })
    if y_true_reg is not None and y_pred_reg is not None:
        df_reg = pd.DataFrame({
            'y_true_amount': y_true_reg,
            'pred_amount': y_pred_reg,
        }, index=range(len(y_true_reg)))
    path = out_dir / 'predictions.csv'
    df.to_csv(path, index=False)
    logger.info(f"Prediksi disimpan: {path}")

def save_feature_importance(shap_values, feature_names, out_dir, source='shap'):
    shap_mean = np.abs(shap_values).mean(axis=0)
    df = pd.DataFrame({'feature': feature_names, 'importance': shap_mean})
    df = df.sort_values('importance', ascending=False).reset_index(drop=True)
    df['rank'] = df.index + 1
    df['source'] = source
    path = out_dir / 'feature_importance.csv'
    df.to_csv(path, index=False)
    logger.info(f"Feature importance disimpan: {path}")
    return df

def save_hyperparams(params_occ, params_reg, out_dir):
    hp = {'stage1_classifier': params_occ, 'stage2_regressor': params_reg}
    path = out_dir / 'best_hyperparameters.json'
    with open(path, 'w') as f:
        json.dump(hp, f, indent=2)
    logger.info(f"Hyperparameter disimpan: {path}")


## Fungsi Optimasi Bayesian (Optuna)
Fungsi untuk menjalankan pencarian hyperparameter menggunakan Optuna dan kalibrasi probabilitas.

In [8]:
def print_callback(study, trial):
    print(f"Trial {trial.number:3d} | Val Score: {trial.value:.5f} | "
          f"Best: {study.best_value:.5f} | Params: {study.best_params}")

def run_optuna_occ(X_train, y_train, X_val, y_val, n_trials):
    def objective(trial):
        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'logloss',
            'tree_method': 'hist',
            'device': 'cuda' if is_kaggle() else 'cpu',
            'max_depth':      trial.suggest_int('max_depth', 3, 8),
            'learning_rate':  trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
            'n_estimators':   trial.suggest_int('n_estimators', 50, 300),
            'subsample':      trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'random_state': CONFIG['SEED'],
            'verbosity': 0,
        }
        clf = xgb.XGBClassifier(**params)
        clf.fit(X_train, y_train, verbose=False)
        prob = clf.predict_proba(X_val)[:, 1]
        return log_loss(y_val, prob, labels=[0, 1])

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, callbacks=[print_callback])
    return study.best_params

def run_optuna_reg(X_train, y_train, X_val, y_val, n_trials):
    def objective(trial):
        params = {
            'objective': 'reg:squarederror',
            'tree_method': 'hist',
            'device': 'cuda' if is_kaggle() else 'cpu',
            'max_depth':      trial.suggest_int('max_depth', 3, 8),
            'learning_rate':  trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
            'n_estimators':   trial.suggest_int('n_estimators', 50, 300),
            'subsample':      trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'random_state': CONFIG['SEED'],
            'verbosity': 0,
        }
        reg = xgb.XGBRegressor(**params)
        reg.fit(X_train, y_train, verbose=False)
        pred = reg.predict(X_val)
        return mean_squared_error(y_val, pred)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, callbacks=[print_callback])
    return study.best_params

def calibrate_model(prob_uncal_val, y_val):
    iso   = IsotonicRegression(out_of_bounds='clip')
    platt = LogisticRegression()
    iso.fit(prob_uncal_val, y_val)
    platt.fit(prob_uncal_val.reshape(-1, 1), y_val)

    prob_iso   = iso.predict(prob_uncal_val)
    prob_platt = platt.predict_proba(prob_uncal_val.reshape(-1, 1))[:, 1]
    brier_iso   = brier_score_loss(y_val, prob_iso)
    brier_platt = brier_score_loss(y_val, prob_platt)

    if brier_iso <= brier_platt:
        best = iso
        method = 'Isotonic'
    else:
        best = platt
        method = 'Platt Scaling'
    logger.info(f"Kalibrasi terpilih: {method} (Brier: {min(brier_iso, brier_platt):.4f})")
    return best, method, prob_iso, prob_platt

def apply_calibrator(cal, prob):
    if hasattr(cal, 'predict_proba'):
        return cal.predict_proba(prob.reshape(-1, 1))[:, 1]
    return cal.predict(prob)


## Analisis SHAP
Fungsi untuk menghitung nilai SHAP menggunakan TreeExplainer pada model XGBoost.

In [9]:
def compute_shap_xgb(model, X, feature_names, n_sample=500):
    X_s = pd.DataFrame(X, columns=feature_names)
    if len(X_s) > n_sample:
        X_s = X_s.sample(n_sample, random_state=CONFIG['SEED'])
    explainer = shap.TreeExplainer(model)
    shap_vals  = explainer.shap_values(X_s)
    return shap_vals, X_s.columns.tolist(), X_s


## Pipeline Utama XGBoost
Fungsi `run_xgb_pipeline(scale, df_feat)` menjalankan pipeline lengkap untuk satu skala waktu:

1. Persiapan target dan pemisahan data
2. Optimasi Optuna Stage 1 (Klasifikasi)
3. Optimasi Optuna Stage 2 (Regresi)
4. Pelatihan model akhir + kalibrasi probabilitas
5. Evaluasi pada data uji
6. Analisis SHAP + Gain Importance
7. Pembuatan semua plot (9 plot terpisah)
8. Ekspor semua file output

In [10]:
def run_xgb_pipeline(scale, df_feat):
    scfg = SCALE_CONFIG[scale]
    out_scale    = OUTPUT_BASE / scale
    out_plots    = out_scale / 'plots'
    out_metrics  = out_scale / 'metrics'
    out_shap     = out_scale / 'shap'
    out_preds    = out_scale / 'predictions'

    logger.info(f"\n{'='*60}")
    logger.info(f"PIPELINE XGBoost — {scfg['label'].upper()}")
    logger.info(f"{'='*60}")

    # -------------------------------------------------------
    # 1. Siapkan target
    # -------------------------------------------------------
    df = prepare_targets(df_feat, scale)
    features = df.drop(columns=['target_amount', 'target_occurrence'])
    targets  = df[['target_amount', 'target_occurrence']]
    feat_names = features.columns.tolist()

    train_mask = (df.index.year >= CONFIG['TRAIN_YEARS'][0]) & (df.index.year <= CONFIG['TRAIN_YEARS'][1])
    val_mask   = df.index.year == CONFIG['VAL_YEAR']
    test_mask  = df.index.year == CONFIG['TEST_YEAR']

    if CONFIG['TEST_MODE']:
        n = len(df)
        tr = int(n * 0.70)
        vl = int(n * 0.85)
        train_mask = pd.Series([False]*n, index=df.index)
        val_mask   = pd.Series([False]*n, index=df.index)
        test_mask  = pd.Series([False]*n, index=df.index)
        train_mask.iloc[:tr] = True
        val_mask.iloc[tr:vl]  = True
        test_mask.iloc[vl:]   = True

    if train_mask.sum() < 10 or val_mask.sum() < 5 or test_mask.sum() < 5:
        logger.warning(f"[{scale}] Data tidak cukup — skip pipeline")
        return None

    X_train, y_train = features[train_mask], targets[train_mask]
    X_val,   y_val   = features[val_mask],   targets[val_mask]
    X_test,  y_test  = features[test_mask],  targets[test_mask]

    rain_mask_tr  = y_train['target_occurrence'] == 1
    rain_mask_val = y_val['target_occurrence']   == 1
    rain_mask_te  = y_test['target_occurrence']  == 1

    scaler = MinMaxScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)
    X_test_s  = scaler.transform(X_test)

    # -------------------------------------------------------
    # 2. Optuna — Stage 1 (Classifier)
    # -------------------------------------------------------
    logger.info(f"[{scale}] Optuna Stage 1 — Klasifikasi ({CONFIG['N_TRIALS']} trials)...")
    p_occ = run_optuna_occ(X_train_s, y_train['target_occurrence'],
                           X_val_s,   y_val['target_occurrence'],
                           CONFIG['N_TRIALS'])
    p_occ.update({'objective': 'binary:logistic', 'tree_method': 'hist',
                  'device': 'cuda' if is_kaggle() else 'cpu',
                  'random_state': CONFIG['SEED'], 'verbosity': 0})

    # -------------------------------------------------------
    # 3. Optuna — Stage 2 (Regressor)
    # -------------------------------------------------------
    logger.info(f"[{scale}] Optuna Stage 2 — Regresi ({CONFIG['N_TRIALS']} trials)...")
    X_train_reg = X_train_s[rain_mask_tr]
    y_train_reg = y_train[rain_mask_tr]['target_amount'].values
    X_val_reg   = X_val_s[rain_mask_val]
    y_val_reg   = y_val[rain_mask_val]['target_amount'].values

    p_reg = None
    if len(X_train_reg) > 5 and len(X_val_reg) > 3:
        p_reg = run_optuna_reg(X_train_reg, y_train_reg,
                               X_val_reg,   y_val_reg,
                               CONFIG['N_TRIALS'])
        p_reg.update({'objective': 'reg:squarederror', 'tree_method': 'hist',
                      'device': 'cuda' if is_kaggle() else 'cpu',
                      'random_state': CONFIG['SEED'], 'verbosity': 0})
    else:
        logger.warning(f"[{scale}] Tidak cukup sampel hujan untuk regresi")

    save_hyperparams(p_occ, p_reg or {}, out_preds)

    # -------------------------------------------------------
    # 4. Latih Model Akhir — Stage 1
    # -------------------------------------------------------
    logger.info(f"[{scale}] Melatih model akhir Stage 1...")
    clf = xgb.XGBClassifier(**p_occ)
    clf.fit(X_train_s, y_train['target_occurrence'],
            eval_set=[(X_val_s, y_val['target_occurrence'])], verbose=50)

    prob_val_uncal  = clf.predict_proba(X_val_s)[:,  1]
    prob_test_uncal = clf.predict_proba(X_test_s)[:, 1]

    cal, cal_method, prob_iso_val, prob_platt_val = calibrate_model(prob_val_uncal, y_val['target_occurrence'].values)
    prob_test_cal = apply_calibrator(cal, prob_test_uncal)
    pred_test_occ = (prob_test_cal >= 0.5).astype(int)
    y_test_occ = y_test['target_occurrence'].values

    # -------------------------------------------------------
    # 5. Latih Model Akhir — Stage 2
    # -------------------------------------------------------
    pred_test_reg = np.zeros(rain_mask_te.sum())
    X_test_reg    = X_test_s[rain_mask_te]
    y_test_reg    = y_test[rain_mask_te]['target_amount'].values

    reg = None
    if p_reg is not None and len(X_train_reg) > 5:
        logger.info(f"[{scale}] Melatih model akhir Stage 2...")
        reg = xgb.XGBRegressor(**p_reg)
        reg.fit(X_train_reg, y_train_reg)
        if len(X_test_reg) > 0:
            pred_test_reg = reg.predict(X_test_reg)
            pred_test_reg = np.maximum(pred_test_reg, 0)

    # -------------------------------------------------------
    # 6. Evaluasi
    # -------------------------------------------------------
    clf_metrics = compute_classification_metrics(
        y_test_occ, prob_test_uncal, prob_test_cal, pred_test_occ)
    reg_metrics = compute_regression_metrics(y_test_reg, pred_test_reg) if len(y_test_reg) > 0 else {}

    logger.info(f"[{scale}] Metrik klasifikasi: {clf_metrics}")
    logger.info(f"[{scale}] Metrik regresi: {reg_metrics}")

    save_classification_metrics(clf_metrics, out_metrics, scale)
    if reg_metrics:
        save_regression_metrics(reg_metrics, out_metrics, scale)

    # -------------------------------------------------------
    # 7. Simpan Prediksi
    # -------------------------------------------------------
    save_predictions(y_test_occ, prob_test_cal, pred_test_occ,
                     y_test_reg, pred_test_reg if len(pred_test_reg)>0 else None,
                     X_test.index.tolist(), out_preds)

    # -------------------------------------------------------
    # 8. SHAP Feature Importance
    # -------------------------------------------------------
    logger.info(f"[{scale}] Menghitung SHAP...")
    try:
        shap_vals, shap_feats, _ = compute_shap_xgb(clf, X_train_s, feat_names)
        save_feature_importance(shap_vals, feat_names, out_shap, source='shap_classifier')
        plot_shap_importance(shap_vals, feat_names, out_shap, scale, title_suffix='Stage 1 Classifier')
    except Exception as e:
        logger.warning(f"[{scale}] SHAP error: {e}")

    try:
        plot_gain_importance(clf, feat_names, out_shap, scale, title_suffix='Stage 1 Classifier')
    except Exception as e:
        logger.warning(f"[{scale}] Gain importance error: {e}")

    if reg is not None:
        try:
            shap_reg_vals, shap_reg_feats, _ = compute_shap_xgb(reg, X_train_reg, feat_names)
            plot_shap_importance(shap_reg_vals, feat_names, out_shap, scale, title_suffix='Stage 2 Regressor')
            plot_gain_importance(reg, feat_names, out_shap, scale, title_suffix='Stage 2 Regressor')
        except Exception as e:
            logger.warning(f"[{scale}] SHAP Regressor error: {e}")

    # -------------------------------------------------------
    # 9. Visualisasi — Section A (Reliabilitas)
    # -------------------------------------------------------
    prob_iso_test   = apply_calibrator(IsotonicRegression(out_of_bounds='clip').fit(prob_val_uncal, y_val['target_occurrence'].values), prob_test_uncal)
    prob_platt_test = apply_calibrator(LogisticRegression().fit(prob_val_uncal.reshape(-1,1), y_val['target_occurrence'].values), prob_test_uncal)

    plot_reliability_diagram(y_test_occ, prob_test_uncal, prob_iso_test, prob_platt_test, out_plots, scale)
    plot_probability_distribution(prob_test_uncal, prob_iso_test, prob_platt_test, out_plots, scale)

    # -------------------------------------------------------
    # 10. Visualisasi — Section B (Klasifikasi)
    # -------------------------------------------------------
    plot_confusion_matrix(y_test_occ, pred_test_occ, out_plots, scale)
    plot_roc_curve(y_test_occ, prob_test_cal, out_plots, scale)
    plot_precision_recall_curve(y_test_occ, prob_test_cal, out_plots, scale)

    # -------------------------------------------------------
    # 11. Visualisasi — Section C (Regresi & Meteorological Validation)
    # -------------------------------------------------------
    if len(y_test_reg) > 0 and len(pred_test_reg) > 0:
        test_reg_index = X_test[rain_mask_te].index
        plot_prediction_vs_observation(y_test_reg, pred_test_reg, out_plots, scale)
        plot_residual_distribution(y_test_reg, pred_test_reg, out_plots, scale)
        plot_timeseries_prediction(y_test_reg, pred_test_reg, test_reg_index, out_plots, scale)
        plot_hexbin_prediction_vs_observation(y_test_reg, pred_test_reg, out_plots, scale)

    # 12. Meteorological Validation (BMKG Category Confusion Matrix & CSI vs Threshold)
    try:
        df_align = pd.DataFrame(index=X_test.index)
        df_align['y_true_amount'] = y_test['target_amount']
        df_align['pred_occ'] = np.nan
        df_align.loc[X_test.index, 'pred_occ'] = pred_test_occ
        df_align['pred_reg'] = np.nan
        if reg is not None and len(pred_test_reg) > 0:
            df_align.loc[test_reg_index, 'pred_reg'] = pred_test_reg
        df_align = df_align.dropna(subset=['pred_occ'])
        df_align['pred_reg'] = df_align['pred_reg'].fillna(0.0)
        df_align['y_pred_amount'] = df_align['pred_reg'] * df_align['pred_occ']
        
        y_true_all = df_align['y_true_amount'].values
        y_pred_all = df_align['y_pred_amount'].values
        
        plot_meteorological_confusion_matrix(y_true_all, y_pred_all, out_plots, scale)
        plot_csi_vs_threshold(y_true_all, y_pred_all, out_plots, scale)
    except Exception as e:
        logger.warning(f"[{scale}] Meteorological validation plotting error: {e}")

    logger.info(f"[{scale}] Pipeline selesai! Output: {out_scale}")

    return {
        'model': 'xgboost',
        'scale': scale,
        **{f'clf_{k}': v for k, v in clf_metrics.items()},
        **{f'reg_{k}': v for k, v in reg_metrics.items()},
    }


## Eksekusi: Pemuatan Data
Memuat dan memproses data cuaca sebelum menjalankan pipeline.

In [11]:
data_path = get_paths()
df_raw  = load_data(data_path)
df_feat = generate_features(df_raw)
feature_names = df_feat.drop(columns=['rain'], errors='ignore').columns.tolist()
logger.info(f"Fitur tersedia: {len(feature_names)} kolom")
df_feat.info()


2026-07-11 16:00:46,494 - INFO - Memuat data dari /kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/open_meteo_jerukagung/cuaca_jerukagung.csv
2026-07-11 16:00:47,836 - INFO - Data dimuat: 232,488 baris, 23 kolom | 2000-01-01 00:00:00 s/d 2026-07-09 23:00:00
2026-07-11 16:00:47,935 - INFO - Fitur tersedia: 33 kolom


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 228006 entries, 2000-01-01 01:00:00 to 2026-07-05 06:00:00
Data columns (total 34 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   rain                                  228006 non-null  float64
 1   temperature_2m                        228006 non-null  float64
 2   wet_bulb_temperature_2m               228006 non-null  float64
 3   relative_humidity_2m                  228006 non-null  float64
 4   dew_point_2m                          228006 non-null  float64
 5   total_column_integrated_water_vapour  228006 non-null  float64
 6   surface_pressure                      228006 non-null  float64
 7   pressure_msl                          228006 non-null  float64
 8   wind_gusts_10m                        228006 non-null  float64
 9   cloud_cover                           228006 non-null  float64
 10  cloud_cover_high                  

## Pipeline Prediksi 1 Jam
Menjalankan pipeline lengkap untuk prediksi curah hujan 1 jam ke depan.

In [12]:
results_1h = run_xgb_pipeline('1h', df_feat)
print("\n[SELESAI] Pipeline 1 Jam")
if results_1h:
    print(f"  ROC-AUC: {results_1h.get('clf_ROC_AUC', 'N/A'):.4f}")
    print(f"  CSI:     {results_1h.get('clf_CSI', 'N/A'):.4f}")


2026-07-11 16:00:47,977 - INFO - 
2026-07-11 16:00:47,978 - INFO - PIPELINE XGBoost — PREDIKSI 1 JAM
2026-07-11 16:00:47,978 - INFO - ============================================================
2026-07-11 16:00:48,030 - INFO - [1h] Dataset: 228,005 baris | Kejadian hujan: 14.1% | Threshold: >=0.5 mm/jam
2026-07-11 16:00:48,160 - INFO - [1h] Optuna Stage 1 — Klasifikasi (30 trials)...


Trial   0 | Val Score: 0.30001 | Best: 0.30001 | Params: {'max_depth': 5, 'learning_rate': 0.001135656826245618, 'n_estimators': 227, 'subsample': 0.9406014290086873, 'colsample_bytree': 0.7734234007906274, 'min_child_weight': 10}
Trial   1 | Val Score: 0.23340 | Best: 0.23340 | Params: {'max_depth': 7, 'learning_rate': 0.0034956592459932565, 'n_estimators': 218, 'subsample': 0.713871594065443, 'colsample_bytree': 0.7078456140415293, 'min_child_weight': 4}
Trial   2 | Val Score: 0.23754 | Best: 0.23340 | Params: {'max_depth': 7, 'learning_rate': 0.0034956592459932565, 'n_estimators': 218, 'subsample': 0.713871594065443, 'colsample_bytree': 0.7078456140415293, 'min_child_weight': 4}
Trial   3 | Val Score: 0.16623 | Best: 0.16623 | Params: {'max_depth': 6, 'learning_rate': 0.022990466377918024, 'n_estimators': 171, 'subsample': 0.748227940832725, 'colsample_bytree': 0.8570919065600728, 'min_child_weight': 7}
Trial   4 | Val Score: 0.31518 | Best: 0.16623 | Params: {'max_depth': 6, 'learn

2026-07-11 16:01:50,163 - INFO - [1h] Optuna Stage 2 — Regresi (30 trials)...


Trial  29 | Val Score: 0.16270 | Best: 0.16264 | Params: {'max_depth': 8, 'learning_rate': 0.03590681042781644, 'n_estimators': 241, 'subsample': 0.9266883188212803, 'colsample_bytree': 0.8198844353266608, 'min_child_weight': 9}
Trial   0 | Val Score: 1.63568 | Best: 1.63568 | Params: {'max_depth': 8, 'learning_rate': 0.01178372262855127, 'n_estimators': 197, 'subsample': 0.7675710787330993, 'colsample_bytree': 0.6691212620829066, 'min_child_weight': 10}
Trial   1 | Val Score: 1.63348 | Best: 1.63348 | Params: {'max_depth': 6, 'learning_rate': 0.09259579304872793, 'n_estimators': 227, 'subsample': 0.9519198035603801, 'colsample_bytree': 0.890361162018434, 'min_child_weight': 6}
Trial   2 | Val Score: 2.10647 | Best: 1.63348 | Params: {'max_depth': 6, 'learning_rate': 0.09259579304872793, 'n_estimators': 227, 'subsample': 0.9519198035603801, 'colsample_bytree': 0.890361162018434, 'min_child_weight': 6}
Trial   3 | Val Score: 1.67340 | Best: 1.63348 | Params: {'max_depth': 6, 'learning_r

2026-07-11 16:02:10,748 - INFO - Hyperparameter disimpan: /kaggle/working/outputs/xgboost/1h/predictions/best_hyperparameters.json
2026-07-11 16:02:10,748 - INFO - [1h] Melatih model akhir Stage 1...


Trial  29 | Val Score: 1.78090 | Best: 1.54693 | Params: {'max_depth': 8, 'learning_rate': 0.02902060243809784, 'n_estimators': 147, 'subsample': 0.9148155747522002, 'colsample_bytree': 0.7375070698982733, 'min_child_weight': 8}
[0]	validation_0-logloss:0.35544
[50]	validation_0-logloss:0.18027
[100]	validation_0-logloss:0.16556
[150]	validation_0-logloss:0.16290
[200]	validation_0-logloss:0.16263
[240]	validation_0-logloss:0.16264


2026-07-11 16:02:14,589 - INFO - Kalibrasi terpilih: Isotonic (Brier: 0.0487)
2026-07-11 16:02:14,591 - INFO - [1h] Melatih model akhir Stage 2...
2026-07-11 16:02:15,666 - INFO - [1h] Metrik klasifikasi: {'Accuracy': 0.9115296803652968, 'Precision': 0.7680865449628127, 'Recall': 0.7244897959183674, 'F1': 0.7456514604529045, 'ROC_AUC': 0.9467965752763841, 'PR_AUC': 0.829487971435257, 'Brier_Uncal': 0.06409333646297455, 'Brier_Cal': 0.06476379185914993, 'CSI': np.float64(0.5944531658817374), 'POD': np.float64(0.7244897959183674), 'FAR': np.float64(0.2319134550371873), 'ETS': np.float64(0.5292376103992931), 'HSS': np.float64(0.6921587682650651)}
2026-07-11 16:02:15,667 - INFO - [1h] Metrik regresi: {'MAE': 0.7667685030142263, 'RMSE': np.float64(1.30451726974401), 'R2': 0.760554424379262, 'NSE': np.float64(0.7605544243792641), 'KGE': np.float64(0.7789972043268307), 'Bias': np.float64(0.036061140157434404), 'Correlation': np.float64(0.873905180490661)}
2026-07-11 16:02:15,674 - INFO - Metr


[SELESAI] Pipeline 1 Jam
  ROC-AUC: 0.9468
  CSI:     0.5945


## Pipeline Prediksi 3 Jam
Menjalankan pipeline lengkap untuk prediksi curah hujan 3 jam ke depan.

In [13]:
results_3h = run_xgb_pipeline('3h', df_feat)
print("\n[SELESAI] Pipeline 3 Jam")
if results_3h:
    print(f"  ROC-AUC: {results_3h.get('clf_ROC_AUC', 'N/A'):.4f}")
    print(f"  CSI:     {results_3h.get('clf_CSI', 'N/A'):.4f}")


2026-07-11 16:02:19,473 - INFO - 
2026-07-11 16:02:19,475 - INFO - PIPELINE XGBoost — PREDIKSI 3 JAM
2026-07-11 16:02:19,476 - INFO - ============================================================
2026-07-11 16:02:19,592 - INFO - [3h] Dataset: 76,003 baris | Kejadian hujan: 19.8% | Threshold: >=1.0 mm/3jam
2026-07-11 16:02:19,615 - INFO - [3h] Optuna Stage 1 — Klasifikasi (30 trials)...


Trial   0 | Val Score: 0.37533 | Best: 0.37533 | Params: {'max_depth': 7, 'learning_rate': 0.0028376063981304226, 'n_estimators': 106, 'subsample': 0.712180342188027, 'colsample_bytree': 0.9788411068463535, 'min_child_weight': 6}
Trial   1 | Val Score: 0.40134 | Best: 0.37533 | Params: {'max_depth': 7, 'learning_rate': 0.0028376063981304226, 'n_estimators': 106, 'subsample': 0.712180342188027, 'colsample_bytree': 0.9788411068463535, 'min_child_weight': 6}
Trial   2 | Val Score: 0.32893 | Best: 0.32893 | Params: {'max_depth': 4, 'learning_rate': 0.002544849393232953, 'n_estimators': 269, 'subsample': 0.8224202071149727, 'colsample_bytree': 0.8817845738922907, 'min_child_weight': 6}
Trial   3 | Val Score: 0.36753 | Best: 0.32893 | Params: {'max_depth': 4, 'learning_rate': 0.002544849393232953, 'n_estimators': 269, 'subsample': 0.8224202071149727, 'colsample_bytree': 0.8817845738922907, 'min_child_weight': 6}
Trial   4 | Val Score: 0.27994 | Best: 0.27994 | Params: {'max_depth': 8, 'learn

2026-07-11 16:02:58,872 - INFO - [3h] Optuna Stage 2 — Regresi (30 trials)...


Trial  29 | Val Score: 0.24244 | Best: 0.24055 | Params: {'max_depth': 7, 'learning_rate': 0.028661796340616056, 'n_estimators': 199, 'subsample': 0.6748047481116302, 'colsample_bytree': 0.6050398262952251, 'min_child_weight': 3}
Trial   0 | Val Score: 12.90986 | Best: 12.90986 | Params: {'max_depth': 4, 'learning_rate': 0.04569971808512578, 'n_estimators': 268, 'subsample': 0.9004041252235304, 'colsample_bytree': 0.8601432804926055, 'min_child_weight': 10}
Trial   1 | Val Score: 11.31547 | Best: 11.31547 | Params: {'max_depth': 8, 'learning_rate': 0.0018820823463119739, 'n_estimators': 266, 'subsample': 0.9930007571810205, 'colsample_bytree': 0.7478556214453251, 'min_child_weight': 6}
Trial   2 | Val Score: 11.59402 | Best: 11.31547 | Params: {'max_depth': 8, 'learning_rate': 0.0018820823463119739, 'n_estimators': 266, 'subsample': 0.9930007571810205, 'colsample_bytree': 0.7478556214453251, 'min_child_weight': 6}
Trial   3 | Val Score: 11.08824 | Best: 11.08824 | Params: {'max_depth':

2026-07-11 16:03:16,672 - INFO - Hyperparameter disimpan: /kaggle/working/outputs/xgboost/3h/predictions/best_hyperparameters.json
2026-07-11 16:03:16,672 - INFO - [3h] Melatih model akhir Stage 1...


Trial  29 | Val Score: 11.18454 | Best: 9.52901 | Params: {'max_depth': 6, 'learning_rate': 0.005723891974138383, 'n_estimators': 238, 'subsample': 0.6973698219615767, 'colsample_bytree': 0.9554714072586683, 'min_child_weight': 3}
[0]	validation_0-logloss:0.44949
[50]	validation_0-logloss:0.27291
[100]	validation_0-logloss:0.24665
[150]	validation_0-logloss:0.24172
[198]	validation_0-logloss:0.24055


2026-07-11 16:03:18,231 - INFO - Kalibrasi terpilih: Isotonic (Brier: 0.0723)
2026-07-11 16:03:18,233 - INFO - [3h] Melatih model akhir Stage 2...
2026-07-11 16:03:19,154 - INFO - [3h] Metrik klasifikasi: {'Accuracy': 0.8565068493150685, 'Precision': 0.7835951134380453, 'Recall': 0.603494623655914, 'F1': 0.6818526955201215, 'ROC_AUC': 0.912269973513599, 'PR_AUC': 0.808484591351313, 'Brier_Uncal': 0.09848055988550186, 'Brier_Cal': 0.10132943838834763, 'CSI': np.float64(0.5172811059907834), 'POD': np.float64(0.603494623655914), 'FAR': np.float64(0.2164048865619546), 'ETS': np.float64(0.41966979216866324), 'HSS': np.float64(0.5912216974449852)}
2026-07-11 16:03:19,154 - INFO - [3h] Metrik regresi: {'MAE': 2.2875236649026154, 'RMSE': np.float64(4.303279553863854), 'R2': 0.5510769749248683, 'NSE': np.float64(0.5510769749248697), 'KGE': np.float64(0.4791043601341737), 'Bias': np.float64(-0.2524869126017376), 'Correlation': np.float64(0.7873462330994676)}
2026-07-11 16:03:19,156 - INFO - Metr


[SELESAI] Pipeline 3 Jam
  ROC-AUC: 0.9123
  CSI:     0.5173


## Pipeline Prediksi Harian
Menjalankan pipeline lengkap untuk prediksi curah hujan harian (24 jam) ke depan.

In [14]:
results_daily = run_xgb_pipeline('daily', df_feat)
print("\n[SELESAI] Pipeline Harian")
if results_daily:
    print(f"  ROC-AUC: {results_daily.get('clf_ROC_AUC', 'N/A'):.4f}")
    print(f"  CSI:     {results_daily.get('clf_CSI', 'N/A'):.4f}")


2026-07-11 16:03:22,504 - INFO - 
2026-07-11 16:03:22,505 - INFO - PIPELINE XGBoost — PREDIKSI HARIAN
2026-07-11 16:03:22,505 - INFO - ============================================================
2026-07-11 16:03:22,620 - INFO - [daily] Dataset: 9,501 baris | Kejadian hujan: 70.0% | Threshold: >=1.0 mm/hari
2026-07-11 16:03:22,628 - INFO - [daily] Optuna Stage 1 — Klasifikasi (30 trials)...


Trial   0 | Val Score: 0.37173 | Best: 0.37173 | Params: {'max_depth': 7, 'learning_rate': 0.009646508314548578, 'n_estimators': 234, 'subsample': 0.770727352826837, 'colsample_bytree': 0.8118874264575915, 'min_child_weight': 4}
Trial   1 | Val Score: 0.56819 | Best: 0.37173 | Params: {'max_depth': 7, 'learning_rate': 0.009646508314548578, 'n_estimators': 234, 'subsample': 0.770727352826837, 'colsample_bytree': 0.8118874264575915, 'min_child_weight': 4}
Trial   2 | Val Score: 0.38201 | Best: 0.37173 | Params: {'max_depth': 7, 'learning_rate': 0.009646508314548578, 'n_estimators': 234, 'subsample': 0.770727352826837, 'colsample_bytree': 0.8118874264575915, 'min_child_weight': 4}
Trial   3 | Val Score: 0.65602 | Best: 0.37173 | Params: {'max_depth': 7, 'learning_rate': 0.009646508314548578, 'n_estimators': 234, 'subsample': 0.770727352826837, 'colsample_bytree': 0.8118874264575915, 'min_child_weight': 4}
Trial   4 | Val Score: 0.34627 | Best: 0.34627 | Params: {'max_depth': 3, 'learning_

2026-07-11 16:03:33,337 - INFO - [daily] Optuna Stage 2 — Regresi (30 trials)...


Trial  29 | Val Score: 0.51618 | Best: 0.34627 | Params: {'max_depth': 3, 'learning_rate': 0.05522788688263106, 'n_estimators': 249, 'subsample': 0.6307266639737282, 'colsample_bytree': 0.8882251038649118, 'min_child_weight': 3}
Trial   0 | Val Score: 116.98623 | Best: 116.98623 | Params: {'max_depth': 3, 'learning_rate': 0.0019155071993725425, 'n_estimators': 55, 'subsample': 0.8565431588003751, 'colsample_bytree': 0.7258614660874804, 'min_child_weight': 7}
Trial   1 | Val Score: 94.99537 | Best: 94.99537 | Params: {'max_depth': 4, 'learning_rate': 0.010299640194098333, 'n_estimators': 98, 'subsample': 0.8432168172778316, 'colsample_bytree': 0.855188049684134, 'min_child_weight': 3}
Trial   2 | Val Score: 105.32956 | Best: 94.99537 | Params: {'max_depth': 4, 'learning_rate': 0.010299640194098333, 'n_estimators': 98, 'subsample': 0.8432168172778316, 'colsample_bytree': 0.855188049684134, 'min_child_weight': 3}
Trial   3 | Val Score: 100.23079 | Best: 94.99537 | Params: {'max_depth': 4,

2026-07-11 16:03:47,155 - INFO - Hyperparameter disimpan: /kaggle/working/outputs/xgboost/daily/predictions/best_hyperparameters.json
2026-07-11 16:03:47,156 - INFO - [daily] Melatih model akhir Stage 1...


Trial  28 | Val Score: 102.82186 | Best: 94.99537 | Params: {'max_depth': 4, 'learning_rate': 0.010299640194098333, 'n_estimators': 98, 'subsample': 0.8432168172778316, 'colsample_bytree': 0.855188049684134, 'min_child_weight': 3}
Trial  29 | Val Score: 100.06494 | Best: 94.99537 | Params: {'max_depth': 4, 'learning_rate': 0.010299640194098333, 'n_estimators': 98, 'subsample': 0.8432168172778316, 'colsample_bytree': 0.855188049684134, 'min_child_weight': 3}
[0]	validation_0-logloss:0.66926
[50]	validation_0-logloss:0.37575
[100]	validation_0-logloss:0.35372
[150]	validation_0-logloss:0.34811
[200]	validation_0-logloss:0.34630
[248]	validation_0-logloss:0.34627


2026-07-11 16:03:47,479 - INFO - Kalibrasi terpilih: Isotonic (Brier: 0.0985)
2026-07-11 16:03:47,480 - INFO - [daily] Melatih model akhir Stage 2...
2026-07-11 16:03:47,735 - INFO - [daily] Metrik klasifikasi: {'Accuracy': 0.8027397260273973, 'Precision': 0.8766666666666667, 'Recall': 0.8825503355704698, 'F1': 0.8795986622073578, 'ROC_AUC': 0.786662325954122, 'PR_AUC': 0.943849906896169, 'Brier_Uncal': 0.12656092643737793, 'Brier_Cal': 0.14453083276748657, 'CSI': np.float64(0.7850746268656716), 'POD': np.float64(0.8825503355704698), 'FAR': np.float64(0.12333333333333334), 'ETS': np.float64(0.2006083650190114), 'HSS': np.float64(0.3341778565999493)}
2026-07-11 16:03:47,736 - INFO - [daily] Metrik regresi: {'MAE': 8.882148587463686, 'RMSE': np.float64(20.212230927903523), 'R2': 0.08462103666335574, 'NSE': np.float64(0.08462103666335641), 'KGE': np.float64(-0.05681328400554286), 'Bias': np.float64(-2.7188943462883866), 'Correlation': np.float64(0.380085175920304)}
2026-07-11 16:03:47,739


[SELESAI] Pipeline Harian
  ROC-AUC: 0.7867
  CSI:     0.7851


## Laporan Perbandingan Akhir
Membandingkan kinerja model XGBoost untuk semua skala waktu dan mengekspor laporan ringkasan.

In [15]:
def generate_comparison_report(results_list, output_base):
    valid = [r for r in results_list if r is not None]
    if not valid:
        logger.warning("Tidak ada hasil untuk dibandingkan")
        return

    df_cmp = pd.DataFrame(valid)

    # Kolom ringkasan
    clf_cols = ['clf_ROC_AUC','clf_F1','clf_CSI','clf_POD','clf_FAR','clf_ETS','clf_HSS',
                'clf_Accuracy','clf_Precision','clf_Recall','clf_PR_AUC','clf_Brier_Cal']
    reg_cols = ['reg_RMSE','reg_MAE','reg_R2','reg_NSE','reg_KGE','reg_Bias','reg_Correlation']
    all_cols = ['model','scale'] + [c for c in clf_cols+reg_cols if c in df_cmp.columns]
    df_export = df_cmp[all_cols].copy()

    path_csv = output_base.parent / 'model_comparison_xgboost.csv'
    df_export.to_csv(path_csv, index=False)
    logger.info(f"Laporan perbandingan disimpan: {path_csv}")

    # Plot ringkasan
    scales_avail = df_export['scale'].tolist()
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, metric, label in zip(
        axes,
        ['clf_ROC_AUC', 'clf_CSI', 'reg_RMSE'],
        ['ROC-AUC', 'CSI', 'RMSE (mm)']
    ):
        vals = [df_export[df_export['scale']==s][metric].values[0]
                if metric in df_export.columns and s in df_export['scale'].values else 0
                for s in ['1h','3h','daily']]
        ax.bar(['1 Jam','3 Jam','Harian'], vals)
        ax.set_title(label, fontsize=13, fontweight='bold')
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.3, axis='y')
        for i, v in enumerate(vals):
            ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)

    fig.suptitle('Perbandingan Kinerja XGBoost Multi-Skala Waktu', fontsize=15, fontweight='bold')
    plt.tight_layout()
    path_fig = output_base.parent / 'summary_comparison_xgboost.png'
    fig.savefig(path_fig, dpi=150, bbox_inches='tight')
    plt.close(fig)
    logger.info(f"Ringkasan plot disimpan: {path_fig}")

    display(df_export)
    return df_export

# Buat laporan perbandingan
all_results = [results_1h, results_3h, results_daily]
df_comparison = generate_comparison_report(all_results, OUTPUT_BASE)
print("\n=== PIPELINE XGBOOST V2 SELESAI ===")
print(f"Output tersimpan di: {OUTPUT_BASE}")


2026-07-11 16:03:50,767 - INFO - Laporan perbandingan disimpan: /kaggle/working/outputs/model_comparison_xgboost.csv
2026-07-11 16:03:51,085 - INFO - Ringkasan plot disimpan: /kaggle/working/outputs/summary_comparison_xgboost.png


,model,scale,clf_ROC_AUC,clf_F1,clf_CSI,clf_POD,clf_FAR,clf_ETS,clf_HSS,clf_Accuracy,...,clf_Recall,clf_PR_AUC,clf_Brier_Cal,reg_RMSE,reg_MAE,reg_R2,reg_NSE,reg_KGE,reg_Bias,reg_Correlation
0,xgboost,1h,0.946797,0.745651,0.594453,0.724490,0.231913,0.529238,0.692159,0.911530,...,0.724490,0.829488,0.064764,1.304517,0.766769,0.760554,0.760554,0.778997,0.036061,0.873905
1,xgboost,3h,0.912270,0.681853,0.517281,0.603495,0.216405,0.419670,0.591222,0.856507,...,0.603495,0.808485,0.101329,4.303280,2.287524,0.551077,0.551077,0.479104,-0.252487,0.787346
2,xgboost,daily,0.786662,0.879599,0.785075,0.882550,0.123333,0.200608,0.334178,0.802740,...,0.882550,0.943850,0.144531,20.212231,8.882149,0.084621,0.084621,-0.056813,-2.718894,0.380085



=== PIPELINE XGBOOST V2 SELESAI ===
Output tersimpan di: /kaggle/working/outputs/xgboost
